# 14회차 실습 노트북 — 데이터 보안 및 품질 관리

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/14_security_quality.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/14_security_quality.html](https://adminmrobo.github.io/fintech_data_analysis/14_security_quality.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 준식별자 조합으로 한 사람이 특정되는가

In [2]:
ins = load('Revolution/insurance.csv'); ins['bmi_r'] = ins.bmi.round(1)
for cols in (['age', 'sex', 'bmi_r', 'children', 'region'], ['age', 'sex', 'children', 'region'], ['age', 'sex', 'region']):
    g = ins.groupby(cols).size()
    print(cols, ' 혼자인 묶음', (g == 1).sum(), '/ 전체 묶음', len(g))

['age', 'sex', 'bmi_r', 'children', 'region']  혼자인 묶음 1330 / 전체 묶음 1334
['age', 'sex', 'children', 'region']  혼자인 묶음 569 / 전체 묶음 866
['age', 'sex', 'region']  혼자인 묶음 2 / 전체 묶음 370


## 2. 대결 재현: 덜 묶은 비식별 vs 크게 묶은 비식별 (k-익명성)

In [3]:
def kanon(w, region, children):
    d = ins.copy(); d['ab'] = (d.age // w) * w
    cols = ['ab', 'sex'] + (['region'] if region else []) + (['children'] if children else [])
    g = d.groupby(cols).size()
    return dict(k=int(g.min()), 묶음=len(g), 위험인원=int(g[g < 5].sum()))
print('나이 5살 단위 + 자녀 공개:', kanon(5, 0, 1))
print('나이 20살 단위, 지역·자녀 삭제:', kanon(20, 0, 0))

나이 5살 단위 + 자녀 공개: {'k': 1, '묶음': 106, '위험인원': 61}
나이 20살 단위, 지역·자녀 삭제: {'k': 56, '묶음': 8, '위험인원': 0}


## 3. 원자료 품질 점검

In [4]:
t = load('Revolution/travel insurance.csv')
print('성별 결측', t.Gender.isna().sum(), '/', len(t))
print('여행 기간 음수', (t.Duration < 0).sum(), '  나이 100세 초과', (t.Age > 100).sum(), '  순매출 음수', (t['Net Sales'] < 0).sum())
s = load('Revolution6/Financial_Sentiment_Analysis.csv')
print('감성 문장 중복', s.Sentence.duplicated().sum(), '  같은 문장에 다른 라벨', s.groupby('Sentence').Sentiment.nunique().gt(1).sum())

성별 결측 45107 / 63326
여행 기간 음수 5   나이 100세 초과 984   순매출 음수 678
감성 문장 중복 520   같은 문장에 다른 라벨 514


**AI 판단 연습** — 나이 118세 고객이 여럿이라면 오류일까요, 기본값(코드)일까요? 지우기 전에 누구에게 무엇을 물어야 할까요?